# Part 4: vmap — Vectorization

**Goal**: Transform single-example functions into batched functions automatically. By the end, you'll use `vmap` to eliminate Python loops over batch dimensions, compose it with `grad` for per-example gradients, and understand how it relates to XLA-level vectorization.

---

## Table of Contents

1. **The Problem vmap Solves** — Python loops vs. vectorization
2. **vmap Basics** — `in_axes`, `out_axes`
3. **vmap Patterns** — vmap+grad, jit(vmap(f)), nested vmap
4. **Common Misconceptions**
5. **Summary — What To Do Next**

---

> **Prerequisites**: This notebook builds on **Notebooks 02–03**. You should be comfortable with `grad` and `jit`.

In [1]:
# @title Setup { display-mode: "form" }

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"JAX version: {jax.__version__}")
print(f"Devices:     {jax.devices()}")

JAX version: 0.9.2
Devices:     [CpuDevice(id=0)]


In [2]:
# @title Reconstruct MLP from Notebooks 01-02 { display-mode: "form" }

def init_mlp_params(key, layer_sizes):
    """Initialize MLP parameters with He initialization."""
    params = []
    for i in range(len(layer_sizes) - 1):
        key, w_key, b_key = jax.random.split(key, 3)
        fan_in = layer_sizes[i]
        fan_out = layer_sizes[i + 1]
        w = jax.random.normal(w_key, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in)
        b = jnp.zeros(fan_out)
        params.append((w, b))
    return params

def mlp_forward(params, x):
    """Forward pass for a SINGLE input vector."""
    for i, (w, b) in enumerate(params):
        x = x @ w + b
        if i < len(params) - 1:
            x = jnp.maximum(x, 0)
    return x

key = jax.random.PRNGKey(42)
params = init_mlp_params(key, [8, 64, 32, 1])
print("MLP reconstructed: 8 → 64 → 32 → 1")

MLP reconstructed: 8 → 64 → 32 → 1


---

# 1. The Problem vmap Solves

Our `mlp_forward` from Notebooks 01–02 processes one input at a time. To handle a batch of inputs, we used a Python list comprehension:

```python
preds = jnp.array([mlp_forward(params, xi) for xi in x_batch])
```

This is slow — it loops in Python, dispatching one operation at a time. There are three ways to handle batching. Let's compare them.

In [3]:
key = jax.random.PRNGKey(0)
x_batch = jax.random.normal(key, (1000, 8))

# --- Approach 1: Python loop ---
def batch_loop(params, x_batch):
    return jnp.array([mlp_forward(params, xi) for xi in x_batch])

# --- Approach 2: Manual batching (rewrite forward pass for batches) ---
def mlp_forward_batched(params, x):
    """Forward pass rewritten to handle batch dimension."""
    for i, (w, b) in enumerate(params):
        x = x @ w + b  # Works because @ broadcasts over batch dim
        if i < len(params) - 1:
            x = jnp.maximum(x, 0)
    return x

# --- Approach 3: vmap ---
batch_vmap = jax.vmap(mlp_forward, in_axes=(None, 0))
# in_axes=(None, 0) means: don't map over params, map over axis 0 of x

# Warm up JIT
batch_loop_jit = jax.jit(batch_loop)
batch_manual_jit = jax.jit(mlp_forward_batched)
batch_vmap_jit = jax.jit(batch_vmap)

_ = batch_loop_jit(params, x_batch)
_ = batch_manual_jit(params, x_batch)
_ = batch_vmap_jit(params, x_batch)

# Time them
results = {}
for name, fn in [("Python loop", batch_loop_jit),
                 ("Manual batch", batch_manual_jit),
                 ("vmap", batch_vmap_jit)]:
    start = time.time()
    for _ in range(100):
        out = fn(params, x_batch)
        out[0].block_until_ready() if isinstance(out, jnp.ndarray) else out.block_until_ready() if hasattr(out, 'block_until_ready') else jax.tree.map(lambda x: x.block_until_ready(), out)
    elapsed = time.time() - start
    results[name] = elapsed / 100

print(f"{'Method':<20} | {'Time (ms)':>10} | {'vs vmap':>10}")
print("-" * 48)
vmap_time = results["vmap"]
for name, t in results.items():
    print(f"{name:<20} | {t*1000:>8.3f} ms | {t/vmap_time:>8.1f}x")

# Verify all give the same result
out_loop = batch_loop_jit(params, x_batch)
out_manual = batch_manual_jit(params, x_batch)
out_vmap = batch_vmap_jit(params, x_batch)
print(f"\nAll equal? loop≈manual: {jnp.allclose(out_loop.squeeze(), out_manual.squeeze(), atol=1e-5)}, "
      f"manual≈vmap: {jnp.allclose(out_manual.squeeze(), out_vmap.squeeze(), atol=1e-5)}")

Method               |  Time (ms) |    vs vmap
------------------------------------------------
Python loop          |    0.287 ms |      4.0x
Manual batch         |    0.072 ms |      1.0x
vmap                 |    0.071 ms |      1.0x

All equal? loop≈manual: True, manual≈vmap: True


The Python loop is slow because it can't be fully optimized by XLA. Manual batching works but requires rewriting your function. **vmap gives you the speed of manual batching with the clarity of writing for a single example.**

> **Why is vmap faster than a Python loop?** A Python loop dispatches one operation at a time to the device. `vmap` rewrites the function to include the batch dimension *before* it ever reaches XLA — so XLA sees the entire batch at once and can fuse, vectorize, and schedule it optimally.

> **vmap transforms a function that works on one example into a function that works on a batch — automatically.**

---

# 2. vmap Basics

## in_axes and out_axes

[`vmap`](https://jax.readthedocs.io/en/latest/_autosummary/jax.vmap.html) needs to know **which arguments to map over** and **along which axis**. That's what `in_axes` controls.

| `in_axes` value | Meaning |
|---|---|
| `0` | Map over axis 0 (batch dimension) |
| `1` | Map over axis 1 |
| `None` | Don't map — broadcast this argument to all batch elements |
| Tuple | Per-argument specification, e.g., `(0, None)` |

`out_axes` works symmetrically for *outputs*: it controls which axis of each output array the batch results are stacked along. The default `0` is almost always correct. Use `out_axes=1` if you want the batch dimension to be the second axis in the output.

In [4]:
# Our forward pass takes (params, x_single)
# We want to map over x (axis 0) but broadcast params (None)
batched_forward = jax.vmap(mlp_forward, in_axes=(None, 0))

# Now it handles batches natively
x_batch = jax.random.normal(jax.random.PRNGKey(0), (5, 8))
preds = batched_forward(params, x_batch)
print(f"Input batch shape:  {x_batch.shape}")
print(f"Output batch shape: {preds.shape}")

# Compare with single-example calls
for i in range(3):
    single = mlp_forward(params, x_batch[i])
    print(f"  Sample {i}: vmap={preds[i].item():.6f}, single={single.item():.6f}, "
          f"match={jnp.isclose(preds[i], single)}")


Input batch shape:  (5, 8)
Output batch shape: (5, 1)


  Sample 0: vmap=1.322807, single=1.322807, match=[ True]
  Sample 1: vmap=-1.880675, single=-1.880675, match=[ True]
  Sample 2: vmap=0.387344, single=0.387345, match=[ True]


In [5]:
# By default, vmap stacks outputs along axis 0
# out_axes lets you change this

def compute_stats(x):
    """For a single vector, return (mean, std)."""
    return jnp.mean(x), jnp.std(x)

x_batch = jax.random.normal(jax.random.PRNGKey(0), (100, 8))

# Map over axis 0, stack results along axis 0 (default)
means, stds = jax.vmap(compute_stats)(x_batch)
print(f"Means shape: {means.shape}")
print(f"Stds shape:  {stds.shape}")
print(f"First 5 means: {means[:5]}")

Means shape: (100,)
Stds shape:  (100,)
First 5 means: [ 0.29234707  0.40531012  0.05107511 -0.05372689 -0.02388942]


---

# 3. vmap Patterns

## vmap + grad: Per-Example Gradients

One of vmap's most powerful compositions: compute the gradient for **each example independently**. This is useful for differential privacy (per-example gradient clipping), understanding which examples contribute most to the update, and more.

> **What is per-example gradient clipping?** In standard training, gradients are averaged over the whole batch before the parameter update — individual samples' contributions are invisible. Per-example gradients let you clip each sample's gradient norm independently before averaging, a core step in DP-SGD (Differentially Private SGD) for privacy-preserving training.

In [6]:
def single_loss(params, x, y):
    """Loss for a SINGLE example."""
    pred = mlp_forward(params, x)
    return jnp.mean((pred - y) ** 2)

# grad w.r.t. params for a single example
single_grad_fn = jax.grad(single_loss)

# vmap over examples: per-example gradients
per_example_grad_fn = jax.vmap(single_grad_fn, in_axes=(None, 0, 0))

# Generate data
key = jax.random.PRNGKey(0)
x_batch = jax.random.normal(key, (32, 8))
y_batch = jax.random.normal(jax.random.PRNGKey(1), (32,))

per_example_grads = per_example_grad_fn(params, x_batch, y_batch)

# Each gradient has a batch dimension
print("Per-example gradient shapes:")
for i, (dw, db) in enumerate(per_example_grads):
    print(f"  Layer {i}: dw {dw.shape}, db {db.shape}")

print(f"\nFirst dimension (32) = number of examples.")
print(f"Each example has its own independent gradient.")

Per-example gradient shapes:
  Layer 0: dw (32, 8, 64), db (32, 64)
  Layer 1: dw (32, 64, 32), db (32, 32)
  Layer 2: dw (32, 32, 1), db (32, 1)

First dimension (32) = number of examples.
Each example has its own independent gradient.


## vmap + jit: The Standard Pattern

[`jit`](https://jax.readthedocs.io/en/latest/_autosummary/jax.jit.html)([`vmap`](https://jax.readthedocs.io/en/latest/_autosummary/jax.vmap.html)(f)) is the standard composition. JIT compiles the vmapped function, getting both vectorization and compilation benefits.

In [7]:
# The standard pattern: vmap for batching, jit for speed
batched_forward_fast = jax.jit(jax.vmap(mlp_forward, in_axes=(None, 0)))

x_large = jax.random.normal(jax.random.PRNGKey(0), (10000, 8))

# Warm up
_ = batched_forward_fast(params, x_large)

start = time.time()
for _ in range(100):
    out = batched_forward_fast(params, x_large)
    out.block_until_ready()
elapsed = (time.time() - start) / 100

print(f"jit(vmap(forward)) on 10,000 examples: {elapsed*1000:.3f} ms")
print(f"That's {10000/elapsed:.0f} examples/second")

jit(vmap(forward)) on 10,000 examples: 0.266 ms
That's 37611342 examples/second


## Nested vmap: Multiple Batch Dimensions

You can stack vmap calls to handle multiple batch dimensions. A common use case: computing pairwise distances between two sets of points.

In [8]:
def single_distance(x, y):
    """Euclidean distance between two vectors."""
    return jnp.sqrt(jnp.sum((x - y) ** 2))

# Pairwise: for every x_i and every y_j, compute distance(x_i, y_j)
# Inner vmap: fix x, map over all y's
# Outer vmap: map over all x's
pairwise_distance = jax.vmap(jax.vmap(single_distance, in_axes=(None, 0)), in_axes=(0, None))

# Test
a = jnp.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])  # 3 points
b = jnp.array([[1.0, 1.0], [2.0, 2.0]])                 # 2 points

D = pairwise_distance(a, b)
print(f"Points A: {a.shape[0]}, Points B: {b.shape[0]}")
print(f"Distance matrix shape: {D.shape}")
print(f"Distance matrix:\n{D}")
print(f"\nD[0,0] = dist([0,0], [1,1]) = √2 = {jnp.sqrt(2.0):.4f}  ✓")

Points A: 3, Points B: 2
Distance matrix shape: (3, 2)
Distance matrix:
[[1.4142135 2.828427 ]
 [1.        2.236068 ]
 [1.        2.236068 ]]

D[0,0] = dist([0,0], [1,1]) = √2 = 1.4142  ✓


---

# 4. Common Misconceptions

## Misconception: "vmap is just a parallel for-loop"

vmap is a **transformation** — it rewrites your function's computation graph to include a batch dimension. It doesn't launch threads or processes. The resulting batched function is then compiled and optimized by XLA, which may or may not parallelize the computation depending on the hardware. The benefit of vmap is **clarity** (write for one example) and **optimization opportunity** (XLA sees the full batched graph).

## Misconception: "vmap handles ragged batches"

No. Every element in a vmapped axis must have the **same shape**. If your data has variable-length sequences, you need to pad them to a common length and use masking. We'll address this in Notebook 09 when we discuss static-shape strategies for inference.

---

# 5. Summary — What To Do Next

## Key Takeaways

1. **[`vmap`](https://jax.readthedocs.io/en/latest/_autosummary/jax.vmap.html) transforms a single-example function into a batched function** — write code for one input, then `vmap` it. No manual reshaping, no rewriting.

2. **`in_axes` controls mapping**: `0` maps over the batch dimension, `None` broadcasts. Use `(None, 0)` to broadcast params and map over data.

3. **`vmap(grad(f))` gives per-example gradients** — one of JAX's most powerful compositions, useful for differential privacy and gradient analysis.

4. **`jit(vmap(f))` is the standard pattern** — JIT compiles the vmapped function, getting both vectorization and compilation benefits.

## What's Next

In **Notebook 05: Pytrees and Model Parameters**, we'll:
- Restructure model parameters as proper pytrees (nested dicts)
- Use `jax.tree.map` for clean parameter updates
- Understand how this pytree foundation is what Flax builds on top of

---

# Exercises

1. **Batched loss**: Write a per-example loss function and use `vmap` to compute the loss for each sample in a batch. Verify the mean matches calling the batched version directly.

2. **in_axes exploration**: Write a function `f(x, scale)` and use `vmap` to map over `x` while broadcasting `scale`. Then flip it — map over `scale` and broadcast `x`.

3. **Nested vmap**: Compute a pairwise cosine similarity matrix for a set of vectors using nested `vmap`. Verify against a direct `jnp.dot` computation.